# 03. 데이터 전처리 (Data Preprocessing)

## 프로젝트: 반도체 수율 예측 (Semiconductor Yield Prediction)

### 분석 목적

본 Notebook에서는 SECOM 반도체 제조 데이터를 머신러닝 모델 학습에
적합한 형태로 전처리한다.

EDA에서 다음과 같은 주요 데이터 문제를 확인하였다.

- 심각한 Pass/Fail 클래스 불균형
- 다수의 센서 변수에 존재하는 결측값
- 모든 값이 결측이거나 변화가 없는 비정보 변수
- 590개의 센서 변수를 가진 높은 데이터 차원

본 단계에서는 모델 평가의 신뢰성을 유지하고 데이터 누수(Data Leakage)를
방지하기 위해 Training/Test 데이터를 먼저 분리한 후,
Training Data만을 이용하여 전처리 기준을 결정한다.

### 주요 전처리 과정

1. Feature와 Target을 분리한다.
2. Training/Test 데이터를 Stratified Split한다.
3. Training Data를 기준으로 고결측 변수를 식별한다.
4. 모든 값이 결측이거나 정보량이 없는 변수를 제거한다.
5. 나머지 결측값에 대한 Imputation 전략을 설계한다.
6. 이후 모델링에서 사용할 Pipeline 구조를 준비한다.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [2]:
DATA_PATH = Path("../data/processed/uci-secom.csv")

df = pd.read_csv(DATA_PATH)

df["Time"] = pd.to_datetime(df["Time"])

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1567, 592)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,...,feature_542,feature_543,feature_544,feature_545,feature_546,feature_547,feature_548,feature_549,feature_550,feature_551,feature_552,feature_553,feature_554,feature_555,feature_556,feature_557,feature_558,feature_559,feature_560,feature_561,feature_562,feature_563,feature_564,feature_565,feature_566,feature_567,feature_568,feature_569,feature_570,feature_571,feature_572,feature_573,feature_574,feature_575,feature_576,feature_577,feature_578,feature_579,feature_580,feature_581,feature_582,feature_583,feature_584,feature_585,feature_586,feature_587,feature_588,feature_589,Time,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,-0.0034,0.9455,202.4396,0.0,7.9558,414.8710,10.0433,0.9680,192.3963,12.5190,1.4026,-5419.00,2916.50,-4043.75,751.00,0.8955,1.7730,3.0490,64.2333,2.0222,0.1632,3.5191,83.3971,9.5126,50.6170,64.2588,49.3830,66.3141,86.9555,117.5132,61.29,4.515,70.0,352.7173,10.1841,130.3691,723.3092,1.3072,141.2282,1.0,...,0.1096,0.0078,0.0026,7.116,1.0616,395.570,75.752,0.4234,12.93,0.78,0.1827,5.7349,0.3363,39.8842,3.2687,1.0297,1.0344,0.4385,0.1039,42.3877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,533.8500,2.1113,8.95,0.3157,3.0624,0.1026,1.6765,14.9509,NaN,NaN,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,2008-07-19 11:55:00,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,-0.0148,0.9627,200.5470,0.0,10.1548,414.7347,9.2599,0.9701,191.2872,12.4608,1.3825,-5441.50,2604.25,-3498.75,-1640.25,1.2973,2.0143,7.3900,68.4222,2.2667,0.2102,3.4171,84.9052,9.7997,50.6596,64.2828,49.3404,64.9193,87.5241,118.1188,78.25,2.773,70.0,352.2445,10.0373,133.1727,724.8264,1.2887,145.8445,1.0,...,0.1096,0.0078,0.0026,7.116,1.3526,408.798,74.640,0.7193,16.00,1.33,0.2829,7.1196,0.4989,53.1836,3.9139,1.7819,0.9634,0.1745,0.0375,18.1087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,535.0164,2.4335,5.92,0.2653,2.0111,0.0772,1.1065,10.9003,0.0096,0.0201,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,2008-07-19 12:32:00,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,0.0013,0.9615,202.0179,0.0,9.5157,416.7075,9.3144,0.9674,192.7035,12.5404,1.4123,-5447.75,2701.75,-4047.00,-1916.50,1.3122,2.0295,7.5788,67.1333,2.3333,0.1734,3.5986,84.7569,8.6590,50.1530,64.1114,49.8470,65.8389,84.7327,118.6128,14.37,5.434,70.0,364.3782,9.8783,131.8027,734.7924,1.2992,141.0845,1.0,...,0.1096,0.0078,0.0026,7.116,0.7942,411.136,74.654,0.1832,16.16,0.85,0.0857,7.1619,0.3752,23.0713,3.9306,1.1386,1.5021,0.3718,0.1233,24.7524,267.064,0.9032,1.10,0.6219,0.4122,0.2562,0.4119,68.8489,535.0245,2.0293,11.21,0.1882,4.0923,0.0640,2.0952,9.2721,0.0584,0.0484,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,2008-07-19 13:17:00,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,-0.0033,0.9629,201.8482,0.0,9.6052,422.2894,9.6924,0.9687,192.1557,12.4782,1.4011,-5468.25,2648.25,-4515.00,-1657.25,1.3137,2.0038,7.3145,62.9333,2.6444,0.2071,3.3813,84.9105,8.6789,50.5100,64.1125,49.4900,65.1951,86.6867,117.0442,76.90,1.279,70.0,363.0273,9.9305,131.8027,733.8778,1.3027,142.5427,1.0,...,0.1096,0.0078,0.0026,7.116,1.1650,372.822,72.442,1.8804,131.68,39.33,0.6812,56.9303,17.4781,161.4081,35.3198,54.2917,1.1613,0.7288,0.2710,62.7572,268.228,0.6511,7.32,0.1630,3.5611,0.0670,2.7290,25.0363,530.5682,2.0253,9.33,0.1738,2.8971,0.0525,1.7585,8.5831,0.0202,0.0149,0.0044,73.8432,0.4990,0.0103,0.002

## 1. Feature와 Target 분리

머신러닝 모델 학습을 위해 입력 변수(Feature)와 예측 대상(Target)을 분리한다.

SECOM 데이터에서:

- `feature_0` ~ `feature_589`: 반도체 공정 센서 변수
- `Pass/Fail`: 예측 대상
- `Time`: 데이터 수집 시간

으로 구성되어 있다.

본 모델의 목적은 센서 측정값을 이용하여 Fail 여부를 예측하는 것이므로
`Time`은 모델의 입력 변수에서 제외하고, 별도로 보관한다.

또한 모델링의 편의를 위해 Target 값을 다음과 같이 변환한다.

- Pass (`-1`) → `0`
- Fail (`1`) → `1`

따라서 이후 모델 평가에서 `1`은 우리가 검출하고자 하는 Fail 클래스를 의미한다.

In [3]:
feature_cols = [
    col for col in df.columns
    if col.startswith("feature_")
]

X = df[feature_cols].copy()

y = (
    df["Pass/Fail"]
    .replace({
        -1: 0,
        1: 1
    })
    .astype(int)
)

time_data = df["Time"].copy()

In [4]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nTarget ratio (%):")
print(
    y.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

X shape: (1567, 590)
y shape: (1567,)

Target distribution:
Pass/Fail
0    1463
1     104
Name: count, dtype: int64

Target ratio (%):
Pass/Fail
0    93.36
1     6.64
Name: proportion, dtype: float64


## 2. Training/Test 데이터 분리

데이터 전처리를 수행하기 전에 전체 데이터를 Training Data와 Test Data로 분리한다.

이 순서를 지키는 이유는 결측률, 중앙값, Feature 제거 기준 등의 정보를
전체 데이터에서 미리 계산할 경우 Test Data의 정보가 학습 과정에 포함되는
Data Leakage가 발생할 수 있기 때문이다.

본 프로젝트에서는 전체 데이터의 80%를 Training Data,
20%를 최종 Test Data로 사용한다.

또한 Fail 샘플의 비율이 약 6.64%로 매우 낮기 때문에
`stratify=y`를 사용하여 Training/Test Data에서도
Pass/Fail 비율이 최대한 동일하게 유지되도록 한다.

`random_state=42`를 지정하여 동일한 코드를 실행했을 때
항상 동일한 데이터 분할이 생성되도록 한다.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:
print("Training set")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTest set")
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Training set
X_train: (1253, 590)
y_train: (1253,)

Test set
X_test : (314, 590)
y_test : (314,)


### 클래스 비율 검증

Stratified Split이 정상적으로 수행되었는지 확인하기 위해
전체 데이터, Training Data, Test Data의 Fail 비율을 비교한다.

세 데이터셋에서 비슷한 Fail 비율이 유지된다면
클래스 불균형을 고려한 데이터 분할이 정상적으로 수행된 것으로 볼 수 있다.

In [7]:
print(
    f"Overall Fail ratio : {y.mean() * 100:.2f}%"
)

print(
    f"Train Fail ratio   : {y_train.mean() * 100:.2f}%"
)

print(
    f"Test Fail ratio    : {y_test.mean() * 100:.2f}%"
)

Overall Fail ratio : 6.64%
Train Fail ratio   : 6.62%
Test Fail ratio    : 6.69%


In [8]:
split_summary = pd.DataFrame({
    "Dataset": [
        "Full",
        "Train",
        "Test"
    ],

    "Total": [
        len(y),
        len(y_train),
        len(y_test)
    ],

    "Pass": [
        (y == 0).sum(),
        (y_train == 0).sum(),
        (y_test == 0).sum()
    ],

    "Fail": [
        (y == 1).sum(),
        (y_train == 1).sum(),
        (y_test == 1).sum()
    ],

    "Fail Rate (%)": [
        y.mean() * 100,
        y_train.mean() * 100,
        y_test.mean() * 100
    ]
})

split_summary["Fail Rate (%)"] = (
    split_summary["Fail Rate (%)"]
    .round(2)
)

split_summary

,Dataset,Total,Pass,Fail,Fail Rate (%)
0,Full,1567,1463,104,6.64
1,Train,1253,1170,83,6.62
2,Test,314,293,21,6.69


## 3. Training Data 기반 고결측 변수 식별

EDA에서는 전체 데이터의 결측값 분포를 탐색적으로 확인하였다.

그러나 실제 전처리 기준을 결정할 때 전체 데이터를 사용할 경우
Test Data의 정보가 학습 과정에 반영될 수 있다.

따라서 본 단계에서는 **Training Data만을 이용하여**
각 센서 변수의 결측률을 계산한다.

EDA 결과를 참고하여 결측률이 40% 이상인 센서 변수를
고결측 변수(High-Missing Feature) 후보로 정의한다.

이 기준으로 선택된 Feature 목록은 이후 Test Data에도
동일하게 적용한다.

In [9]:
train_missing_ratio = (
    X_train
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

train_missing_ratio.head(20)

feature_292    90.662410
feature_293    90.662410
feature_158    90.662410
feature_157    90.662410
feature_220    85.634477
feature_85     85.634477
feature_358    85.634477
feature_492    85.634477
feature_384    66.320830
feature_382    66.320830
feature_383    66.320830
feature_245    66.320830
feature_109    66.320830
feature_244    66.320830
feature_516    66.320830
feature_246    66.320830
feature_110    66.320830
feature_517    66.320830
feature_111    66.320830
feature_518    66.320830
dtype: float64

In [10]:
high_missing_features = (
    train_missing_ratio[
        train_missing_ratio >= 40
    ]
    .index
    .tolist()
)

print(
    "Features with >= 40% missing:",
    len(high_missing_features)
)

print("\nHigh-missing features:")
print(high_missing_features)

Features with >= 40% missing: 32

High-missing features:
['feature_292', 'feature_293', 'feature_158', 'feature_157', 'feature_220', 'feature_85', 'feature_358', 'feature_492', 'feature_384', 'feature_382', 'feature_383', 'feature_245', 'feature_109', 'feature_244', 'feature_516', 'feature_246', 'feature_110', 'feature_517', 'feature_111', 'feature_518', 'feature_578', 'feature_580', 'feature_581', 'feature_579', 'feature_346', 'feature_72', 'feature_73', 'feature_345', 'feature_247', 'feature_385', 'feature_519', 'feature_112']


In [11]:
all_missing_features_train = [
    col for col in X_train.columns
    if X_train[col].isna().all()
]

print(
    "All-missing features:",
    len(all_missing_features_train)
)

print(all_missing_features_train)

All-missing features: 0
[]


In [12]:
single_value_features_train = [
    col for col in X_train.columns
    if X_train[col].nunique(dropna=True) == 1
]

print(
    "Single-value features:",
    len(single_value_features_train)
)

print(single_value_features_train)

Single-value features: 116
['feature_5', 'feature_13', 'feature_42', 'feature_49', 'feature_52', 'feature_69', 'feature_97', 'feature_141', 'feature_149', 'feature_178', 'feature_179', 'feature_186', 'feature_189', 'feature_190', 'feature_191', 'feature_192', 'feature_193', 'feature_194', 'feature_226', 'feature_229', 'feature_230', 'feature_231', 'feature_232', 'feature_233', 'feature_234', 'feature_235', 'feature_236', 'feature_237', 'feature_240', 'feature_241', 'feature_242', 'feature_243', 'feature_256', 'feature_257', 'feature_258', 'feature_259', 'feature_260', 'feature_261', 'feature_262', 'feature_263', 'feature_264', 'feature_265', 'feature_266', 'feature_276', 'feature_284', 'feature_313', 'feature_314', 'feature_315', 'feature_322', 'feature_325', 'feature_326', 'feature_327', 'feature_328', 'feature_329', 'feature_330', 'feature_364', 'feature_369', 'feature_370', 'feature_371', 'feature_372', 'feature_373', 'feature_374', 'feature_375', 'feature_378', 'feature_379', 'feat

### 제거 대상 Feature 통합

다음 조건 중 하나 이상에 해당하는 센서 변수를 제거 대상으로 정의한다.

1. Training Data에서 결측률이 40% 이상인 변수
2. Training Data의 모든 값이 결측인 변수
3. Training Data에서 하나의 값만 가지는 변수

여러 조건에 동시에 해당하는 Feature가 존재할 수 있으므로
중복을 제거하여 하나의 Feature 목록으로 통합한다.

In [13]:
features_to_remove = sorted(
    set(
        high_missing_features
        + all_missing_features_train
        + single_value_features_train
    )
)

print(
    "Total features to remove:",
    len(features_to_remove)
)

Total features to remove: 148


In [14]:
print(
    "Original features:",
    X_train.shape[1]
)

print(
    "Features to remove:",
    len(features_to_remove)
)

print(
    "Remaining features:",
    X_train.shape[1] - len(features_to_remove)
)

Original features: 590
Features to remove: 148
Remaining features: 442


## 4. 제거 대상 Feature 적용

앞서 Training Data만을 이용하여 다음 조건에 해당하는 센서 변수를
제거 대상으로 선정하였다.

- 결측률이 40% 이상인 변수
- 모든 값이 결측인 변수
- 하나의 값만 가지는 비정보 변수

이제 Training Data에서 결정한 동일한 Feature 목록을
Training Data와 Test Data에 모두 적용한다.

중요한 점은 Test Data를 이용하여 별도의 제거 기준을 다시 계산하지 않는다는 것이다.

이를 통해 Test Data의 정보가 전처리 기준 결정 과정에 포함되는
Data Leakage를 방지한다.

In [16]:
X_train_filtered = X_train.drop(
    columns=features_to_remove
).copy()

X_test_filtered = X_test.drop(
    columns=features_to_remove
).copy()

print("Before filtering")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nAfter filtering")
print("X_train_filtered:", X_train_filtered.shape)
print("X_test_filtered :", X_test_filtered.shape)

Before filtering
X_train: (1253, 590)
X_test : (314, 590)

After filtering
X_train_filtered: (1253, 442)
X_test_filtered : (314, 442)


In [17]:
same_columns = X_train_filtered.columns.equals(
    X_test_filtered.columns
)

print(
    "Train/Test columns identical:",
    same_columns
)

assert same_columns, \
    "Training and Test feature columns do not match."

Train/Test columns identical: True


## 5. Feature 제거 후 결측치 확인

고결측 및 비정보 변수를 제거한 이후에도
일부 센서 변수에는 결측값이 남아 있을 수 있다.

머신러닝 모델에 데이터를 입력하기 전에
남아 있는 결측값의 규모를 확인하고 적절한 대체(Imputation) 전략을 적용한다.

In [18]:
train_missing_after_filter = (
    X_train_filtered
    .isna()
    .sum()
    .sum()
)

test_missing_after_filter = (
    X_test_filtered
    .isna()
    .sum()
    .sum()
)

print(
    "Remaining missing values in Train:",
    train_missing_after_filter
)

print(
    "Remaining missing values in Test :",
    test_missing_after_filter
)

Remaining missing values in Train: 6479
Remaining missing values in Test : 1529


## 6. 결측값 대체 (Median Imputation)

고결측 센서 변수를 제거한 이후에도 일부 변수에는 결측값이 남아 있다.

본 프로젝트에서는 남아 있는 센서 결측값을 각 변수의
중앙값(Median)으로 대체한다.

평균(Mean)은 극단값의 영향을 상대적으로 크게 받을 수 있는 반면,
중앙값은 이상치가 존재하는 데이터에서도 비교적 안정적으로
대표값을 계산할 수 있다는 장점이 있다.

또한 Data Leakage를 방지하기 위해 중앙값은
Training Data에서만 계산한다.

- Training Data → `fit_transform()`
- Test Data → `transform()`

즉 Test Data의 중앙값은 계산하지 않으며,
Training Data에서 학습된 동일한 중앙값을 Test Data에도 적용한다.

In [19]:
from sklearn.impute import SimpleImputer

median_imputer = SimpleImputer(
    strategy="median"
)

In [20]:
X_train_imputed_array = (
    median_imputer.fit_transform(
        X_train_filtered
    )
)

X_test_imputed_array = (
    median_imputer.transform(
        X_test_filtered
    )
)

In [21]:
X_train_imputed = pd.DataFrame(
    X_train_imputed_array,
    columns=X_train_filtered.columns,
    index=X_train_filtered.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed_array,
    columns=X_test_filtered.columns,
    index=X_test_filtered.index
)

In [22]:
print(
    "X_train_imputed:",
    X_train_imputed.shape
)

print(
    "X_test_imputed :",
    X_test_imputed.shape
)

X_train_imputed.head()

X_train_imputed: (1253, 442)
X_test_imputed : (314, 442)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_50,feature_51,feature_53,feature_54,...,feature_525,feature_526,feature_527,feature_539,feature_540,feature_541,feature_542,feature_543,feature_544,feature_545,feature_546,feature_547,feature_548,feature_549,feature_550,feature_551,feature_552,feature_553,feature_554,feature_555,feature_556,feature_557,feature_558,feature_559,feature_560,feature_561,feature_562,feature_563,feature_564,feature_565,feature_566,feature_567,feature_568,feature_569,feature_570,feature_571,feature_572,feature_573,feature_574,feature_575,feature_576,feature_577,feature_582,feature_583,feature_584,feature_585,feature_586,feature_587,feature_588,feature_589
1198,3075.32,2491.07,2185.1000,1201.0491,0.7821,105.8489,0.1208,1.4002,-0.0151,-0.0006,0.9738,202.9734,3.0143,399.1464,10.2629,0.9824,192.7104,12.5949,1.4024,-5631.75,2667.50,-5768.50,-100.50,1.2700,2.0172,7.3590,77.3444,2.2111,0.1616,3.4600,84.3569,8.3220,50.4013,64.5182,49.5987,66.0788,86.7285,117.9034,82.30,3.178,361.5191,10.1320,135.7700,740.7825,1.2946,143.3245,640.6136,189.1997,4.561,4.782,...,7.5703,1.2962,3.6974,4.3091,0.8188,10.7007,0.1125,0.0087,0.0023,7.7217,0.6008,405.994,73.434,0.3817,19.32,1.69,0.1448,7.9063,0.6527,63.5388,4.7587,2.3014,0.9701,0.1863,0.0458,19.2044,264.272,0.5671,4.980,0.08770,2.09020,0.0382,1.8844,15.4662,529.6464,2.0320,5.81,0.3435,2.0923,0.0925,1.0970,16.9045,0.4974,0.0128,0.0033,2.5767,0.0223,0.0105,0.0034,47.0690
436,3071.58,2489.47,2217.3777,1425.1041,1.7585,106.2556,0.1200,1.5270,0.0066,-0.0124,0.9598,200.2884,10.2077,405.2716,9.4565,0.9596,190.8320,12.4700,1.4232,-5474.25,2997.00,-4648.00,177.50,1.2550,2.0275,7.3687,59.9778,3.0556,0.2512,3.5528,86.0209,9.2328,50.3586,63.7799,49.6414,65.7516,86.5146,117.7358,75.31,2.353,351.0982,9.9294,131.6627,718.6920,1.0375,135.5791,618.3400,49.2902,4.610,4.849,...,4.2740,2.3670,6.8889,3.7386,1.7549,10.7389,0.1096,0.0078,0.0026,7.1160,1.2762,391.906,73.426,0.5744,25.71,0.68,0.2263,11.9517,0.3052,45.0088,6.5602,0.9261,1.0206,0.5414,0.0968,53.0444,264.272,0.5671,4.980,0.08770,2.09020,0.0382,1.8844,15.4662,530.6027,2.3970,7.11,0.2709,2.3928,0.0895,1.3400,11.3021,0.5004,0.0316,0.0066,6.3183,0.0329,0.0055,0.0022,16.6695
635,3017.53,2524.09,2201.0667,880.2317,1.4148,106.5478,0.1211,1.3720,-0.0005,0.0052,0.9675,198.0364,10.1995,415.3266,8.9253,0.9716,189.1111,12.4871,1.3902,-5438.50,2701.25,-4643.50,-642.50,1.2805,1.9990,7.3342,67.4778,2.4667,0.2439,3.4024,85.7419,8.5788,50.3811,64.1013,49.6189,66.2428,86.6582,119.3410,78.00,3.483,363.4464,9.4867,136.4373,736.5305,1.3118,137.8873,637.7709,204.9855,4.622,4.872,...,2.8146,2.0933,7.2716,1.5316,3.4897,5.6896,0.1096,0.0078,0.0026,7.1160,1.3319,404.496,81.990,0.9837,18.41,1.38,0.4284,8.1119,0.5903,73.8618,4.5513,1.6831,0.9674,0.7942,0.1527,82.0961,264.272,0.6510,5.095,0.11895,2.11895,0.0479,1.9812,16.9642,534.1891,2.0627,7.72,0.3563,2.7800,0.1052,1.4452,17.2719,0.4998,0.0097,0.0026,1.9495,0.0328,0.0235,0.0068,71.5333
996,2901.62,2569.45,2223.9000,1745.3724,1.9974,96.7567,0.1241,1.5950,-0.0163,0.0061,0.9835,195.7092,8.4954,399.5080,9.2262,0.9791,186.4830,12.5237,1.4131,-5525.00,2691.50,-366.75,-126.00,1.2680,1.9858,7.3317,69.9444,2.3111,0.1584,3.4554,83.7179,8.7015,50.6219,64.4228,49.3782,66.2507,86.2732,118.5040,80.86,2.617,347.5700,9.9860,137.0636,723.3293,1.2654,144.9200,629.5536,236.2240,4.537,4.751,...,2.5209,0.2943,8.9798,2.5059,1.2825,7.9272,0.1074,0.0102,0.0024,9.4887,0.8698,400.344,72.746,0.3772,21.75,0.86,0.1555,9.5614,0.3132,43.3734,5.4328,1.1822,0.9830,0.1145,0.0324,11.6455,264.272

### Imputation 결과 검증

Median Imputation 적용 후 Training Data와 Test Data에
결측값이 남아 있는지 확인한다.

두 데이터셋의 결측값 개수가 모두 0이라면
결측값 대체가 정상적으로 수행된 것으로 판단한다.

In [23]:
train_missing_final = (
    X_train_imputed
    .isna()
    .sum()
    .sum()
)

test_missing_final = (
    X_test_imputed
    .isna()
    .sum()
    .sum()
)

print(
    "Missing values after imputation - Train:",
    train_missing_final
)

print(
    "Missing values after imputation - Test :",
    test_missing_final
)

Missing values after imputation - Train: 0
Missing values after imputation - Test : 0


In [24]:
assert X_train_imputed.isna().sum().sum() == 0, \
    "Missing values remain in Training Data."

assert X_test_imputed.isna().sum().sum() == 0, \
    "Missing values remain in Test Data."

assert X_train_imputed.shape[1] == 442
assert X_test_imputed.shape[1] == 442

print(
    "Preprocessing validation passed."
)

Preprocessing validation passed.


## 7. 전처리 결과 요약 (Preprocessing Summary)

Training Data만을 기준으로 전처리 규칙을 결정한 결과,
590개의 원본 센서 변수 중 148개 변수가 제거 대상으로 식별되었다.

제거 후 442개의 센서 변수가 유지되었으며,
남아 있는 결측값은 Training Data에서 계산한 중앙값을 이용하여 대체하였다.

이 과정에서 Test Data는 전처리 기준 결정에 사용하지 않았으며,
Training Data에서 학습한 동일한 기준을 적용하였다.

In [25]:
preprocessing_summary = pd.DataFrame({
    "Metric": [
        "Original features",
        "High-missing features",
        "All-missing features",
        "Single-value features",
        "Total removed features",
        "Remaining features",
        "Training samples",
        "Test samples",
        "Train Fail rate (%)",
        "Test Fail rate (%)",
        "Train missing values after imputation",
        "Test missing values after imputation"
    ],
    "Value": [
        X_train.shape[1],
        len(high_missing_features),
        len(all_missing_features_train),
        len(single_value_features_train),
        len(features_to_remove),
        X_train_imputed.shape[1],
        len(X_train),
        len(X_test),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2),
        X_train_imputed.isna().sum().sum(),
        X_test_imputed.isna().sum().sum()
    ]
})

preprocessing_summary

,Metric,Value
0,Original features,590.00
1,High-missing features,32.00
2,All-missing features,0.00
3,Single-value features,116.00
4,Total removed features,148.00
5,Remaining features,442.00
6,Training samples,1253.00
7,Test samples,314.00
8,Train Fail rate (%),6.62
9,Test Fail rate (%),6.69


In [26]:
SUMMARY_PATH = Path(
    "../results/metrics/preprocessing_summary.csv"
)

preprocessing_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Saved to:", SUMMARY_PATH)

Saved to: ..\results\metrics\preprocessing_summary.csv


## 8. 실제 모델링을 위한 Pipeline 설계

앞선 단계에서는 전처리 과정이 정상적으로 작동하는지 확인하기 위해
Training/Test Data에 Feature 제거와 Median Imputation을 직접 적용하였다.

그러나 이후 Cross Validation 및 Hyperparameter Tuning 과정에서는
전처리 과정을 사전에 전체 Training Data에 적용하지 않는다.

Cross Validation에서는 각 Fold마다 다음 과정이 독립적으로 수행되어야 한다.

1. Training Fold에서 전처리 기준을 학습한다.
2. Training Fold의 중앙값을 계산한다.
3. 동일한 기준을 Validation Fold에 적용한다.
4. 필요한 경우 Training Fold에만 SMOTE를 적용한다.
5. 모델을 학습하고 Validation Fold에서 성능을 평가한다.

이를 위해 이후 모델링 단계에서는
전처리와 모델 학습 과정을 Pipeline으로 구성한다.

이 구조를 통해 Validation Data의 정보가 전처리 기준에 반영되는
Data Leakage를 최소화할 수 있다.

### Pipeline 구성 요소

이후 모델링에서는 다음 구성 요소를 필요에 따라 Pipeline에 포함한다.

- `SimpleImputer(strategy="median")`
  - Training Data의 중앙값을 이용한 결측값 대체

- `StandardScaler()`
  - 거리 기반 알고리즘 또는 SMOTE 적용 전에 변수 Scale을 조정

- `SMOTE`
  - Training Data의 소수 클래스(Fail)를 합성하여 클래스 불균형 완화

- Classification Model
  - Logistic Regression, XGBoost 등

특히 SMOTE는 Test Data 또는 Validation Data에는 적용하지 않으며,
Pipeline 내부의 Training 과정에서만 수행한다.

In [27]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [28]:
preprocessing_steps = [
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
]

preprocessing_steps

[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]

In [29]:
smote_pipeline_steps = [
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "smote",
        SMOTE(random_state=42)
    )
]

smote_pipeline_steps

[('imputer', SimpleImputer(strategy='median')),
 ('scaler', StandardScaler()),
 ('smote', SMOTE(random_state=42))]

## 9. 전처리 종합 결과 (Preprocessing Summary)

SECOM 데이터에 대해 모델링 전 데이터 전처리 전략을 설계하였다.

### 데이터 분할

전체 1,567개 샘플을 Stratified Split하여 다음과 같이 분리하였다.

- Training Data: 1,253개
- Test Data: 314개

Fail 비율은:

- 전체 데이터: 약 6.64%
- Training Data: 약 6.62%
- Test Data: 약 6.69%

로 유사하게 유지되었다.


### Feature 품질 검사

Training Data만을 기준으로 센서 변수의 품질을 평가하였다.

- 원본 센서 변수: 590개
- 결측률 40% 이상 변수: 32개
- 모든 값이 결측인 변수: 0개
- 하나의 값만 가지는 변수: 116개
- 최종 제거 대상 변수: 148개
- 유지된 센서 변수: 442개


### 결측값 처리

고결측 및 비정보 변수를 제거한 이후에도 남아 있는 결측값은
Training Data에서 계산한 중앙값(Median)을 이용하여 대체하였다.

Median Imputation 적용 후:

- Training Data 결측값: 0개
- Test Data 결측값: 0개

로 확인되었다.


### Data Leakage 방지 원칙

본 전처리 과정에서는 Test Data를 Feature 제거 기준이나
결측값 대체 기준을 결정하는 데 사용하지 않았다.

모든 전처리 기준은 Training Data에서 학습한 뒤
동일한 기준을 Test Data에 적용하였다.

또한 이후 Cross Validation 및 Hyperparameter Tuning 과정에서는
전처리, Scaling, SMOTE 및 모델 학습 과정을 Pipeline 내부에서 수행하여
Validation Data의 정보가 학습 과정에 포함되지 않도록 설계한다.


## 다음 단계

다음 Notebook에서는 전처리 전략을 기반으로
Baseline Classification Model을 구축한다.

먼저 클래스 불균형 문제를 별도로 보정하지 않은 기본 모델을 학습하여
현재 데이터에서의 기준 성능(Baseline Performance)을 확인한다.

이후 SMOTE 및 XGBoost 등의 방법을 적용한 모델과 비교하여
실제 Fail 검출 성능이 어떻게 개선되는지 평가한다.